In [8]:
import os
import json
import dspy
import numpy as np
import ast
import datasets
from pydantic import BaseModel, Field
from typing import List
import re

cwd = os.getcwd()

### Drafting Code

In [ ]:
#CUDA_VISIBLE_DEVICES=3 vllm serve casperhansen/deepseek-r1-distill-qwen-32b-awq --max-model -len 32768 --gpu-memory-utilization 0.9 --quantization awq_marlin --port 8000

In [2]:
vllm_model = dspy.LM(
    api_base="http://localhost:8000/v1",
    api_key="local",
    model="openai/casperhansen/deepseek-r1-distill-qwen-32b-awq",
    max_tokens=4096
)
dspy.configure(lm=vllm_model)

In [42]:
class IncentiveInstance(BaseModel):
    # using pydantic to ensure our outputs are the right data types
    instrument_text: str = Field(description="The verbatim text (word or phrase) referencing a policy incentive or instrument")
    class_text: str = Field(description="The verbatim text (word or phrase) indicating the class of the policy incentive or instrument")
    label: str = Field(description="The class of policy incentive or instrument")

class IncProc(dspy.Signature):
    """Given a sentence from a policy document, determine if it contains policy incentives or instruments. 
    If so, then for each instance, extract an item containing 
    1) the explicit reference to the policy incentive or instrument (OR 'None' if not explicit)
    2) the explicit text that indicates the class of policy incentive or instrument (OR 'None' if not explicit)
    3) the class of the policy incentive or instrument
    """
    text: str = dspy.InputField(desc="Sentence from a policy document to analyze.")
    output: List[IncentiveInstance] = dspy.OutputField(desc="""Given a sentence from a policy document, 
        FIRST 
        Determine if it mentions a type of policy action, instrument, or incentive.
        - If not, return an empty list. 
        - If it does mention at least one type, make sure the mention is not just a definition of what an incentive type is or an example provided as context for a different discussion. 
            Future tense is usually a good indicator that a mention is relevant. 
        THEN 
        FOR EACH RELEVANT INCENTIVE IN THE SENTENCE                                          
        - extract the text (word or short phrase) that explicitly references or names the incentive (e.g. the title of a scheme) [only if present, this element is not mandatory]
        - extract the text (word or short phrase) that indicates what class of incentive the reference is (e.g. "equipment" indicates the class Supplies) [only if present, this element is not mandatory]
        - classify the incentive type with one of the below labels [this element is mandatory]:
            - Credit: The provision of repayable finance such as loans or insurance to encourage an action.
            - Direct_payment: The provision of non-repayable finance such as cash or grants to encourage an action.
            - Fine: The establishment of a financial penalty for performing a prohibited action or not performing a required action.
            - Supplies: The provision of physical goods, material support, renewables, or equipment to encourage an action.
            - Tax_deduction: The reduction of tax liability to encourage an action.
            - Technical_assistance: The provision of trainings, expertise, or advisory services to encourage an action.
        THEN
        Return a list containing the information for each incentive in the sentence
    """)

test_proc = dspy.ChainOfThought(IncProc)    

In [3]:
ds = datasets.load_dataset("mawaskow/irish_forestry_incentives")

Generating train split: 100%|██████████| 1419/1419 [00:00<00:00, 198681.98 examples/s]


In [36]:
results = []
for entry in list(ds['train'])[:5]:
    pred = test_proc(entry['text'])
    results.append({
        "sentence": entry['text'],
        "label": entry['label'],
        "output": pred.output,
        "reasoning": pred.reasoning
    })

In [37]:
results

[{'sentence': 'Similarly, the On-Farm Ca pital Investments Scheme has provisions for investments in equipment that will allow farmers to reduce the amount of Green House Gas emissions that they produce during their agricultural practices.',
  'label': 'Supplies',
  'output': [IncentiveInstance(instrument_text='provisions for investments in equipment', class_text='equipment', label='Supplies')],
  'reasoning': 'The sentence mentions the "On-Farm Capital Investments Scheme has provisions for investments in equipment." This indicates a policy action or instrument related to providing equipment to farmers. The future tense ("will allow") suggests that it is a relevant mention of an incentive. The text explicitly references the policy instrument (provisions for investments in equipment) and indirectly suggests the class (equipment, a type of supplies).'},
 {'sentence': 'Planting tree belt for ammonia capture at farmyard Low Input Grassland; • The On -Farm Capital Investments Scheme (OFCIS) 

In [27]:
class IncentiveModule(dspy.Module):
    def __init__(self):
        self.extractor = dspy.ChainOfThought(IncProc)
    def forward(self, text):
        return self.extractor(text=text)

program = IncentiveModule()

In [39]:
results = []
for entry in list(ds['train'])[10:15]:
    pred = program(entry['text'])
    results.append({
        "sentence": entry['text'],
        "label": entry['label'],
        "output": pred.output,
        "reasoning": pred.reasoning
    })

In [40]:
results

[{'sentence': 'In these years, the Capital Investment Scheme will contribute to addressing Obj4,N1 and Obj4,N4 by providing support for investments in precision farming equipment a nd in the case of the tillage sector, low disturbance tillage equipment, straw incorporating equipment and equipment to assist the establishment of green cover.',
  'label': 'Supplies',
  'output': [IncentiveInstance(instrument_text='support', class_text='providing support', label='Supplies')],
  'reasoning': 'The sentence mentions the "Capital Investment Scheme" contributing to objectives by "providing support for investments in precision farming equipment" and other related equipment. The phrase "providing support for investments" indicates the presence of a policy incentive. The text explicitly references "support" as the incentive, and the class of incentive is implied to be "providing support" for investments. The classification falls under "Supplies" since it involves the provision of physical goods (e

### Parsing Results

has to be JSONL file like this
{"text": "EU rejects German call to boycott British lamb.", "label": [ [0, 2, "ORG"], [11, 17, "MISC"] ]}
{"text": "Peter Blackburn", "label": [ [0, 15, "PERSON"] ]}
{"text": "President Obama", "label": [ [10, 15, "PERSON"] ]}

In [34]:
with open(f"{cwd}/data/incentive_results_partial.json", "r", encoding="utf-8") as f:
    ds = json.load(f)

In [5]:
example = ds[0]

for example in dataset
    create new entry with text and old_label and outputs
    create label list in entry
    if output
        for output item in example
            if outputitem.instrument_text:
                try to find string ids in sentence
            if outputitem.indicator:
                try to find string ids in sentence
            if not (outputitem.indicator or outputitem.instrument_text):
                string id is just 0:2
            append [string start, string end, outputitem.label] to labellist        

In [ ]:
{"text": example['sentence'],
 "label": []}

{'sentence': 'Similarly, the On-Farm Ca pital Investments Scheme has provisions for investments in equipment that will allow farmers to reduce the amount of Green House Gas emissions that they produce during their agricultural practices.',
 'label': 'Supplies',
 'output': [{'instrument_text': 'On-Farm Capital Investments Scheme',
   'class_text': 'equipment',
   'label': 'Supplies'}],
 'reasoning': 'The sentence mentions "the On-Farm Capital Investments Scheme," which is an explicit reference to a policy incentive. It also provides a specific context for this scheme, pointing out that it allows farmers to invest in equipment to reduce emissions. This future tense ("will allow") suggests that the incentive is actively being provided. The provisions for "equipment" further classify the incentive under Supplies. Therefore, it meets the criteria for inclusion.'}

In [29]:
#match=(re.search(example['output'][0]['class_text'], example['sentence']))
match=(re.search("that", example['sentence']))
i,j = match.span()

In [35]:
span_ds = []
for entry in ds:
    sentence = entry['sentence']
    span_entry = {"text": sentence,
                  "label": [],
                  "old_label": entry['label'],
                  "dspy_lst": entry['output'],
                  "dspy_reasoning": entry['reasoning']
                  }
    if entry['output']:
        for instrumentry in entry['output']:
            insttxt = instrumentry["instrument_text"]
            clstxt = instrumentry["class_text"]
            label = instrumentry["label"]
            if insttxt:
                match=(re.search(insttxt, sentence))
                if match:
                    start, end = match.span()
                    span_entry['label'].append([start, end, label])
            if clstxt:
                match=(re.search(clstxt, sentence))
                if match:
                    start, end = match.span()
                    span_entry['label'].append([start, end, label])
            if not (insttxt or clstxt):
                span_entry.append([0,2,label])
    span_ds.append(span_entry)
print(len(ds), len(span_ds))

600 600


In [32]:
span_ds

[{'text': 'Similarly, the On-Farm Ca pital Investments Scheme has provisions for investments in equipment that will allow farmers to reduce the amount of Green House Gas emissions that they produce during their agricultural practices.',
  'label': [[85, 94, 'Supplies']],
  'old_label': 'Supplies',
  'dspy_lst': [{'instrument_text': 'On-Farm Capital Investments Scheme',
    'class_text': 'equipment',
    'label': 'Supplies'}],
  'dspy_reasoning': 'The sentence mentions "the On-Farm Capital Investments Scheme," which is an explicit reference to a policy incentive. It also provides a specific context for this scheme, pointing out that it allows farmers to invest in equipment to reduce emissions. This future tense ("will allow") suggests that the incentive is actively being provided. The provisions for "equipment" further classify the incentive under Supplies. Therefore, it meets the criteria for inclusion.'},
 {'text': 'Planting tree belt for ammonia capture at farmyard Low Input Grasslan

In [36]:
with open(f"{cwd}/data/firsttry_600.jsonl", "w", encoding="utf-8") as f:
    for sample in span_ds:
        json.dump(sample, f)
        f.write("\n")